# Módulo 10 · Extender pandas y proyecto final

**Curso de pandas intermedio → avanzado** · Dataset: *Olist* (muestras en `data/`)

## Objetivos
1. Crear **accessors propios** (`df.miaccessor.metodo()`) para empaquetar lógica de tu dominio.
2. Presentar tablas con **`Styler`**: formatos, colores, barras y exportación a HTML/Excel.
3. Integrar **todo el curso** en un proyecto final: de los CSV crudos a un reporte ejecutivo.

| Sección | Tema |
|---|---|
| 0 | Setup y carga |
| 1 | Accessors propios |
| 2 | `Styler` |
| 3 | Exportar: Excel con varias hojas y Parquet |
| 🏁 | Proyecto final (ejercicios 3–5) |
| ✍️ | Ejercicios · ✅ Soluciones |

## 0 · Setup y carga

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
  import jinja2  # noqa: F401  (Styler lo necesita)
except ImportError:
  raise ImportError("Styler necesita jinja2: ejecuta `pip install jinja2` en tu entorno")

DATA = Path("data")
SALIDA = Path("salida")
SALIDA.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


def pendiente(*valores) -> bool:
  # True si algún valor sigue siendo `...` (ejercicio sin resolver)
  return any(v is ... for v in valores)


print("pandas", pd.__version__)

pandas 3.1.1


Cargamos **todas** las tablas declarando los dtypes que usaremos (repaso de los Módulos 01 y 07):

In [2]:
FECHAS_PED = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
              "order_delivered_customer_date", "order_estimated_delivery_date"]

orders = pd.read_csv(DATA / "olist_orders_dataset.csv", dtype={"order_status": "category"},
                     parse_dates=FECHAS_PED)
items = pd.read_csv(DATA / "olist_order_items_dataset.csv", dtype={"order_item_id": "int8"},
                    parse_dates=["shipping_limit_date"])
payments = pd.read_csv(DATA / "olist_order_payments_dataset.csv",
                       dtype={"payment_type": "category", "payment_sequential": "int8",
                              "payment_installments": "int8"})
reviews = pd.read_csv(DATA / "olist_order_reviews_dataset.csv",
                      usecols=["review_id", "order_id", "review_score", "review_answer_timestamp"],
                      parse_dates=["review_answer_timestamp"])
customers = pd.read_csv(DATA / "olist_customers_dataset.csv",
                        dtype={"customer_zip_code_prefix": "str", "customer_state": "category"})
sellers = pd.read_csv(DATA / "olist_sellers_dataset.csv",
                      dtype={"seller_zip_code_prefix": "str", "seller_state": "category"})
products = pd.read_csv(DATA / "olist_products_dataset.csv")
traduccion = pd.read_csv(DATA / "product_category_name_translation.csv")

{n: df.shape for n, df in [("orders", orders), ("items", items), ("payments", payments),
                           ("reviews", reviews), ("customers", customers), ("sellers", sellers),
                           ("products", products), ("traduccion", traduccion)]}

{'orders': (300, 8),
 'items': (336, 7),
 'payments': (309, 5),
 'reviews': (302, 4),
 'customers': (300, 5),
 'sellers': (212, 4),
 'products': (296, 9),
 'traduccion': (71, 2)}

## 1 · Accessors propios

pandas trae accessors como `.str`, `.dt` y `.cat`. Puedes **registrar los tuyos** para agrupar lógica de tu dominio bajo un nombre:

```python
@pd.api.extensions.register_series_accessor("br")
class BrasilAccessor:
    def __init__(self, serie):
        self._s = serie
    def metodo(self): ...
```

Después: `serie.br.metodo()`. También existen `register_dataframe_accessor` y `register_index_accessor`.

¿Cuándo vale la pena? Cuando repites la **misma lógica** en muchos notebooks (limpiar ciudades, formatear moneda, validar llaves). Mejor que funciones sueltas porque:

- Se descubren con autocompletado (`serie.br.<Tab>`).
- Se usan **dentro de cadenas** de métodos sin `pipe`.
- Puedes validar el tipo de dato al construir el accessor.

### Un accessor de Series para datos de Brasil

In [3]:
REGIAO = {
  "SP": "Sudeste", "RJ": "Sudeste", "MG": "Sudeste", "ES": "Sudeste",
  "PR": "Sul", "SC": "Sul", "RS": "Sul",
  "BA": "Nordeste", "PE": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "AL": "Nordeste",
  "SE": "Nordeste", "RN": "Nordeste", "PB": "Nordeste", "PI": "Nordeste",
  "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
  "PA": "Norte", "AM": "Norte", "TO": "Norte", "RO": "Norte", "AC": "Norte", "AP": "Norte", "RR": "Norte",
}

# Evita el aviso de "sobrescribir accessor" si re-ejecutas esta celda en Jupyter
warnings.filterwarnings("ignore", message="registration of accessor")


@pd.api.extensions.register_series_accessor("br")
class BrasilAccessor:
  # Utilidades para columnas con datos brasileños: ciudades, CEP, estados y dinero.

  def __init__(self, serie: pd.Series):
    self._s = serie

  def canon(self) -> pd.Series:
    # Llave canónica de texto (Módulo 07): minúsculas, sin acentos ni signos
    return (self._s.astype("str").str.lower()
            .str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("ascii")
            .str.replace(r"[-'’.]", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip())

  def cep(self) -> pd.Series:
    # Prefijo de CEP como texto de 5 dígitos
    return self._s.astype("str").str.zfill(5)

  def regiao(self) -> pd.Series:
    # Región geográfica a partir de la sigla del estado (UF)
    return self._s.map(REGIAO).astype("category")

  def reais(self, decimales: int = 2) -> pd.Series:
    # Formatea números como dinero brasileño: R$ 1.234,56
    fmt = self._s.map(lambda v: f"{v:,.{decimales}f}" if pd.notna(v) else "")
    return ("R$ " + fmt.str.replace(",", "_").str.replace(".", ",").str.replace("_", ".")).where(self._s.notna())

In [4]:
demo = pd.DataFrame({
  "ciudad": ["São Paulo", "MOGI-GUAÇU", "santa bárbara d'oeste"],
  "cep": [1310, 13840, 13450],
  "uf": ["SP", "SP", "SP"],
  "valor": [1234.5, 89.9, np.nan],
})
demo.assign(
  ciudad_canon=lambda d: d["ciudad"].br.canon(),
  cep_txt=lambda d: d["cep"].br.cep(),
  regiao=lambda d: d["uf"].br.regiao(),
  valor_txt=lambda d: d["valor"].br.reais(),
)

,ciudad,cep,uf,valor,ciudad_canon,cep_txt,regiao,valor_txt
0,São Paulo,1310,SP,1234.5,sao paulo,01310,Sudeste,"R$ 1.234,50"
1,MOGI-GUAÇU,13840,SP,89.9,mogi guacu,13840,Sudeste,"R$ 89,90"
2,santa bárbara d'oeste,13450,SP,NaN,santa barbara d oeste,13450,Sudeste,NaN


### Un accessor de DataFrame con validación

Un accessor de DataFrame puede **verificar** que el objeto tenga lo necesario al construirse, y fallar con un mensaje claro si no:

In [5]:
@pd.api.extensions.register_dataframe_accessor("pedidos")
class PedidosAccessor:
  REQUERIDAS = {"order_id", "order_purchase_timestamp",
                "order_delivered_customer_date", "order_estimated_delivery_date"}

  def __init__(self, df: pd.DataFrame):
    faltan = self.REQUERIDAS - set(df.columns)
    if faltan:
      raise AttributeError(f"Faltan columnas para .pedidos: {sorted(faltan)}")
    self._df = df

  def con_tiempos(self) -> pd.DataFrame:
    d = self._df
    entregado = d["order_delivered_customer_date"]
    return d.assign(
      dias_entrega=(entregado - d["order_purchase_timestamp"]).dt.total_seconds() / 86400,
      tarde=(entregado > d["order_estimated_delivery_date"]).astype("float").where(entregado.notna()),
      mes=d["order_purchase_timestamp"].dt.to_period("M"),
    )

  def resumen(self) -> pd.Series:
    t = self.con_tiempos()
    return pd.Series({
      "pedidos": len(t),
      "entregados": t["order_delivered_customer_date"].notna().sum(),
      "dias_entrega_mediana": t["dias_entrega"].median(),
      "pct_tarde": t["tarde"].mean() * 100,
    }).round(2)


orders.pedidos.resumen()

pedidos                 300.00
entregados              293.00
dias_entrega_mediana      9.84
pct_tarde                 7.17
dtype: float64

In [6]:
try:
  items.pedidos.resumen()
except AttributeError as e:
  print("AttributeError:", e)

AttributeError: Faltan columnas para .pedidos: ['order_delivered_customer_date', 'order_estimated_delivery_date', 'order_purchase_timestamp']


In [7]:
# Y encadenable como cualquier método
(orders
 .pedidos.con_tiempos()
 .groupby("mes")["tarde"].mean().mul(100)
 .tail(4)
 .round(1))

mes
2018-05    4.8
2018-06    4.0
2018-07    6.7
2018-08    5.0
Freq: M, Name: tarde, dtype: float64

> 💡 En un proyecto real, los accessors viven en un módulo `.py` que importas al inicio (el registro ocurre al importarlo). En este curso los definimos en el notebook para tener pandas puro y visible.

## 2 · `Styler`

`df.style` devuelve un objeto `Styler` que controla **cómo se ve** la tabla (no cambia los datos). Es ideal para reportes en notebooks, HTML o Excel.

| Método | Para qué |
|---|---|
| `format` | formato por columna (`"{:,.2f}"`, `"{:.1%}"`, funciones) |
| `background_gradient` | mapa de color por valor |
| `bar` | barras dentro de la celda |
| `highlight_max` / `highlight_min` / `highlight_null` | resaltar extremos o faltantes |
| `map` / `apply` | estilos propios por celda / por columna o fila |
| `hide` | ocultar índice o columnas |
| `set_caption`, `set_table_styles`, `set_properties` | título y CSS |
| `to_html`, `to_excel` | exportar con estilos |

> ⚠️ Un `Styler` **no es un DataFrame**: siempre al **final** de la cadena.

In [8]:
cat_en = traduccion.set_index("product_category_name")["product_category_name_english"]

por_cat = (
  items
  .assign(categoria=lambda d: d["product_id"]
          .map(products.set_index("product_id")["product_category_name"])
          .map(cat_en).fillna("sin_categoria"))
  .merge(orders[["order_id", "order_delivered_customer_date", "order_estimated_delivery_date"]],
         on="order_id", validate="m:1")
  .assign(tarde=lambda d: (d["order_delivered_customer_date"] > d["order_estimated_delivery_date"])
                          .astype("float").where(d["order_delivered_customer_date"].notna()))
  .groupby("categoria")
  .agg(items=("order_id", "size"), ingreso=("price", "sum"),
       ticket=("price", "mean"), pct_flete=("freight_value", "sum"), pct_tarde=("tarde", "mean"))
  .assign(pct_flete=lambda d: d["pct_flete"] / d["ingreso"],
          participacion=lambda d: d["ingreso"] / d["ingreso"].sum())
  .sort_values("ingreso", ascending=False)
  .head(10)
)
por_cat

,items,ingreso,ticket,pct_flete,pct_tarde,participacion
categoria,,,,,,
watches_gifts,15,3308.70,220.580000,0.064028,0.000000,0.090403
sports_leisure,34,3143.27,92.449118,0.201930,0.030303,0.085883
health_beauty,29,2715.67,93.643793,0.165488,0.000000,0.074200
bed_bath_table,33,2635.26,79.856364,0.229378,0.090909,0.072003
computers_accessories,27,2524.73,93.508519,0.201566,0.111111,0.068983
auto,12,2009.72,167.476667,0.129137,0.166667,0.054911
furniture_decor,17,1438.02,84.589412,0.300191,0.000000,0.039291
stationery,12,1404.14,117.011667,0.153168,0.000000,0.038365
baby,13,1317.36,101.335385,0.189121,0.166667,0.035994


In [9]:
(por_cat.style
 .format({"ingreso": "R$ {:,.0f}", "ticket": "R$ {:,.2f}",
          "pct_flete": "{:.1%}", "pct_tarde": "{:.1%}", "participacion": "{:.1%}"})
 .bar(subset=["ingreso"], color="#7fb3d5")
 .background_gradient(subset=["pct_tarde"], cmap="Reds")
 .highlight_max(subset=["ticket"], color="#d5f5e3")
 .set_caption("Top 10 categorías por ingreso"))

,items,ingreso,ticket,pct_flete,pct_tarde,participacion
categoria,,,,,,
watches_gifts,15,"R$ 3,309",R$ 220.58,6.4%,0.0%,9.0%
sports_leisure,34,"R$ 3,143",R$ 92.45,20.2%,3.0%,8.6%
health_beauty,29,"R$ 2,716",R$ 93.64,16.5%,0.0%,7.4%
bed_bath_table,33,"R$ 2,635",R$ 79.86,22.9%,9.1%,7.2%
computers_accessories,27,"R$ 2,525",R$ 93.51,20.2%,11.1%,6.9%
auto,12,"R$ 2,010",R$ 167.48,12.9%,16.7%,5.5%
furniture_decor,17,"R$ 1,438",R$ 84.59,30.0%,0.0%,3.9%
stationery,12,"R$ 1,404",R$ 117.01,15.3%,0.0%,3.8%
baby,13,"R$ 1,317",R$ 101.34,18.9%,16.7%,3.6%


### Estilos propios con `map` (por celda) y `apply` (por columna/fila)

In [10]:
def semaforo(v):
  if pd.isna(v):
    return "color: gray"
  return "color: #b03a2e; font-weight: bold" if v > 0.15 else ("color: #b9770e" if v > 0.05 else "color: #1e8449")


def resaltar_top3(col):
  top = col.nlargest(3).index
  return ["background-color: #fef9e7" if i in top else "" for i in col.index]


(por_cat[["items", "ingreso", "pct_tarde"]].style
 .format({"ingreso": "R$ {:,.0f}", "pct_tarde": "{:.1%}"}, na_rep="—")
 .map(semaforo, subset=["pct_tarde"])
 .apply(resaltar_top3, subset=["items"])
 .set_table_styles([{"selector": "th", "props": "background-color: #34495e; color: white;"}]))

,items,ingreso,pct_tarde
categoria,,,
watches_gifts,15,"R$ 3,309",0.0%
sports_leisure,34,"R$ 3,143",3.0%
health_beauty,29,"R$ 2,716",0.0%
bed_bath_table,33,"R$ 2,635",9.1%
computers_accessories,27,"R$ 2,525",11.1%
auto,12,"R$ 2,010",16.7%
furniture_decor,17,"R$ 1,438",0.0%
stationery,12,"R$ 1,404",0.0%
baby,13,"R$ 1,317",16.7%


### Formatos con el accessor

El `format` del `Styler` también acepta funciones, así que podemos reutilizar el formato brasileño:

In [11]:
(por_cat[["ingreso", "ticket"]].head(3).style
 .format(lambda v: pd.Series([v]).br.reais().iloc[0]))

,ingreso,ticket
categoria,,
watches_gifts,"R$ 3.308,70","R$ 220,58"
sports_leisure,"R$ 3.143,27","R$ 92,45"
health_beauty,"R$ 2.715,67","R$ 93,64"


## 3 · Exportar: Excel con varias hojas y Parquet

- `pd.ExcelWriter` escribe **varias hojas** en un archivo. Un `Styler` exporta con sus colores y formatos a Excel con `to_excel`.
- Parquet para que el siguiente análisis lea los datos **con sus dtypes** (Módulo 09).

In [12]:
ruta_xlsx = SALIDA / "demo_reporte.xlsx"

with pd.ExcelWriter(ruta_xlsx, engine="openpyxl") as xl:
  (por_cat.style
   .format({"ingreso": "{:,.2f}", "ticket": "{:,.2f}", "pct_tarde": "{:.1%}"})
   .background_gradient(subset=["pct_tarde"], cmap="Reds")
   .to_excel(xl, sheet_name="categorias"))
  orders.pedidos.resumen().to_frame("valor").to_excel(xl, sheet_name="resumen")

pd.ExcelFile(ruta_xlsx, engine="openpyxl").sheet_names

['categorias', 'resumen']

> 💡 Los formatos de `Styler.format` se exportan como **texto con estilo** sólo en HTML; en Excel se conservan colores y, en general, los valores siguen siendo **números** (que es lo que quieres para que el usuario pueda seguir calculando).

---
# 🏁 Proyecto final: reporte ejecutivo de Olist

Vas a construir, con todo lo del curso, un pipeline que va de los CSV crudos a un reporte. Se divide en tres ejercicios encadenados (3, 4 y 5); cada uno usa el resultado del anterior.

```
CSV crudos ─▶ tabla de hechos por pedido (Ej. 3) ─▶ KPIs (Ej. 4) ─▶ reporte Excel + Parquet (Ej. 5)
```

Módulos que pondrás en práctica: dtypes (01), índices (02), `groupby` (03), reshape (04), uniones con validación (05), fechas (06), texto (07), cadenas y etapas (08), formatos eficientes (09) y presentación (10).

---
# ✍️ Ejercicios

Variables disponibles: todas las tablas cargadas en la sección 0, `REGIAO`, los accessors `.br` y `.pedidos`, `cat_en`, `SALIDA`.

### Ejercicio 1 · Accessor de calidad ⭐

Registra un accessor de **DataFrame** llamado `calidad` con tres métodos:

- `nulos()` → DataFrame indexado por columna con `n` y `pct` de faltantes, **sólo columnas con al menos un faltante**, ordenado por `n` descendente.
- `duplicados(subset=None)` → número de filas duplicadas (contando **todas** las copias, no sólo las repetidas) según `subset`.
- `llave_unica(cols)` → `True`/`False`: si la combinación de columnas identifica cada fila de forma única.

Luego calcula:

- `nulos_productos = products.calidad.nulos()`
- `dup_reviews = reviews.calidad.duplicados(subset=["order_id"])`
- `llave_items = items.calidad.llave_unica(["order_id", "order_item_id"])`

In [13]:
# TU CÓDIGO AQUÍ

nulos_productos = ...
dup_reviews = ...
llave_items = ...

In [14]:
# Comprobación
if pendiente(nulos_productos, dup_reviews, llave_items):
  print("⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba")
else:
  assert hasattr(pd.DataFrame, "calidad"), "Registra el accessor con register_dataframe_accessor"
  assert list(nulos_productos.columns) == ["n", "pct"]
  assert nulos_productos["n"].to_dict() == {"product_category_name": 6, "product_name_lenght": 6,
                                            "product_description_lenght": 6, "product_photos_qty": 6}
  assert dup_reviews == 6, dup_reviews
  assert llave_items is True or llave_items == True
  assert reviews.calidad.llave_unica(["order_id"]) == False
  print("✅ Ejercicio 1 correcto")

⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba


### Ejercicio 2 · Una tabla presentable ⭐⭐

Crea `tabla_estados`: por **región** de cliente (usa `.br.regiao()`), con `pedidos`, `ingreso` (suma de `price` de los ítems), `ticket` (ingreso / pedidos) y `participacion` (% del ingreso total), ordenada por `ingreso` descendente.

Después crea `sty_estados`, un `Styler` sobre `tabla_estados` que:

- formatee `ingreso` y `ticket` como reales con `.br.reais()` (pista: `format` acepta una función),
- formatee `participacion` como porcentaje con 1 decimal,
- agregue barras en `participacion`,
- tenga el título `"Ingreso por región"`.

In [15]:
# TU CÓDIGO AQUÍ
tabla_estados = ...
sty_estados = ...

In [16]:
# Comprobación
if pendiente(tabla_estados, sty_estados):
  print("⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba")
else:
  from pandas.io.formats.style import Styler
  assert list(tabla_estados.columns) == ["pedidos", "ingreso", "ticket", "participacion"]
  assert tabla_estados.index[0] == "Sudeste"
  assert np.isclose(tabla_estados["participacion"].sum(), 1) or np.isclose(tabla_estados["participacion"].sum(), 100)
  assert np.isclose(tabla_estados["ingreso"].sum(), items["price"].sum())
  assert isinstance(sty_estados, Styler)
  html = sty_estados.to_html()
  assert "Ingreso por región" in html and "R$ " in html and "%" in html
  print("✅ Ejercicio 2 correcto")

⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba


### 🏁 Ejercicio 3 · Tabla de hechos por pedido ⭐⭐⭐

Construye `fact`: **una fila por pedido de `orders`** (300 filas), con estas columnas **en este orden**:

| columna | definición |
|---|---|
| `order_id` | |
| `order_status` | |
| `mes` | `Period` mensual de la compra |
| `customer_state` | estado del cliente (`category`) |
| `regiao` | región (`category`, con `.br.regiao()`) |
| `n_items` | número de ítems (0 si no tiene) |
| `valor_items` | suma de `price` (0 si no tiene) |
| `flete` | suma de `freight_value` (0 si no tiene) |
| `pagado` | suma de `payment_value` |
| `metodo_principal` | `payment_type` del pago de **mayor** valor del pedido |
| `cuotas_max` | máximo de `payment_installments` |
| `review_score` | score de la reseña **más reciente** (`NaN` si no hay) |
| `dias_entrega` | días (decimales) de compra a entrega (`NaN` si no se entregó) |
| `tarde` | 1.0 / 0.0 si se entregó tarde / a tiempo; `NaN` si no se entregó |

Reglas: agrega cada tabla "muchos" **antes** de unir, usa `validate` en cada `merge` y, de preferencia, escribe el proceso como etapas con `pipe`.

In [17]:
# TU CÓDIGO AQUÍ
fact = ...

In [18]:
# Comprobación
if pendiente(fact):
  print("⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba")
else:
  COLS = ["order_id", "order_status", "mes", "customer_state", "regiao", "n_items", "valor_items",
          "flete", "pagado", "metodo_principal", "cuotas_max", "review_score", "dias_entrega", "tarde"]
  assert list(fact.columns) == COLS, list(fact.columns)
  assert len(fact) == len(orders) and fact["order_id"].is_unique
  assert isinstance(fact["mes"].dtype, pd.PeriodDtype)
  assert isinstance(fact["regiao"].dtype, pd.CategoricalDtype)
  assert fact["n_items"].sum() == len(items) and (fact["n_items"] == 0).sum() == 1
  assert np.isclose(fact["valor_items"].sum(), items["price"].sum())
  assert np.isclose(fact["pagado"].sum(), payments["payment_value"].sum())
  assert fact["metodo_principal"].astype(str).value_counts().to_dict() == {"credit_card": 237, "boleto": 54, "voucher": 6, "debit_card": 3}
  assert fact["review_score"].isna().sum() == 1
  assert np.isclose(fact["review_score"].mean(), 4.060201, atol=1e-4), fact["review_score"].mean()
  assert fact["tarde"].sum() == 21 and fact["tarde"].isna().sum() == 7
  print("✅ Ejercicio 3 correcto")

⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba


### 🏁 Ejercicio 4 · KPIs ⭐⭐⭐

A partir de `fact`:

1. `kpi_mes`: una fila por mes (`PeriodIndex`) con:
   `pedidos`, `ingreso` (suma de `valor_items`), `ticket` (`ingreso / pedidos`), `pct_tarde` (% sobre entregados), `score` (media de `review_score`), y `crec_ingreso` (% de cambio del ingreso vs mes anterior). Debe incluir **todos** los meses entre el primero y el último, aunque no tengan pedidos (con `pedidos` = 0 e `ingreso` = 0).
2. `kpi_region_trim`: tabla **ancha** con regiones en filas, trimestres (`"2017Q1"`, … como texto) en columnas e `ingreso` en las celdas (0 donde no hay), más una columna `"Total"`; ordenada por `"Total"` descendente.
3. `peor_mes_score`: el mes (como texto `"AAAA-MM"`) con menor `score` **entre los meses con al menos 10 pedidos**.

In [19]:
# TU CÓDIGO AQUÍ
kpi_mes = ...
kpi_region_trim = ...
peor_mes_score = ...

In [20]:
# Comprobación
if pendiente(kpi_mes, kpi_region_trim, peor_mes_score):
  print("⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba")
else:
  assert isinstance(kpi_mes.index, pd.PeriodIndex) and len(kpi_mes) == 20
  assert list(kpi_mes.columns) == ["pedidos", "ingreso", "ticket", "pct_tarde", "score", "crec_ingreso"]
  assert kpi_mes["pedidos"].sum() == len(fact)
  assert np.isclose(kpi_mes["ingreso"].sum(), fact["valor_items"].sum())
  assert np.isclose(kpi_mes.loc["2018-04", "crec_ingreso"], 30.9549, atol=0.01), kpi_mes.loc["2018-04", "crec_ingreso"]
  assert kpi_region_trim.columns[-1] == "Total" and kpi_region_trim["Total"].is_monotonic_decreasing
  assert kpi_region_trim.index[0] == "Sudeste"
  assert "2018Q1" in kpi_region_trim.columns
  assert np.isclose(kpi_region_trim["Total"].sum(), fact["valor_items"].sum())
  assert peor_mes_score == "2017-11", peor_mes_score
  print("✅ Ejercicio 4 correcto")

⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba


### 🏁 Ejercicio 5 · El reporte ⭐⭐⭐

1. Guarda `fact` en `SALIDA / "fact_pedidos.parquet"` y verifica que al releerlo conserva los dtypes de `mes`, `customer_state` y `regiao` (guarda el DataFrame releído en `fact_releido`).
2. Crea `SALIDA / "reporte_olist.xlsx"` con **tres hojas, en este orden**:
   - `"kpis_mensuales"`: `kpi_mes` con estilo (formatos y un gradiente en `pct_tarde`). Convierte el índice a texto antes de exportar.
   - `"region_trimestre"`: `kpi_region_trim` con un gradiente sobre las columnas de trimestres.
   - `"top_categorias"`: el top 10 de categorías por ingreso (puedes reutilizar `por_cat`).
3. `hojas`: la lista de nombres de hojas del archivo generado (léelo con `pd.ExcelFile(..., engine="openpyxl")`; pandas 3.1 avisa que el lector por defecto cambiará a `calamine`, así que conviene declararlo).

In [21]:
# TU CÓDIGO AQUÍ
fact_releido = ...
hojas = ...

In [22]:
# Comprobación
if pendiente(fact_releido, hojas):
  print("⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba")
else:
  assert (SALIDA / "fact_pedidos.parquet").exists()
  assert isinstance(fact_releido["mes"].dtype, pd.PeriodDtype)
  assert isinstance(fact_releido["regiao"].dtype, pd.CategoricalDtype)
  assert fact_releido.equals(fact)
  assert hojas == ["kpis_mensuales", "region_trimestre", "top_categorias"], hojas
  _k = pd.read_excel(SALIDA / "reporte_olist.xlsx", sheet_name="kpis_mensuales", index_col=0, engine="openpyxl")
  assert len(_k) == len(kpi_mes) and np.isclose(_k["ingreso"].sum(), kpi_mes["ingreso"].sum())
  print("✅ Ejercicio 5 correcto")

⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba


---
# ✅ Soluciones

### Solución 1

In [23]:
@pd.api.extensions.register_dataframe_accessor("calidad")
class CalidadAccessor:
  def __init__(self, df: pd.DataFrame):
    self._df = df

  def nulos(self) -> pd.DataFrame:
    n = self._df.isna().sum()
    return (pd.DataFrame({"n": n, "pct": n / len(self._df) * 100})
              .query("n > 0")
              .sort_values("n", ascending=False))

  def duplicados(self, subset=None) -> int:
    return int(self._df.duplicated(subset=subset, keep=False).sum())

  def llave_unica(self, cols) -> bool:
    return not self._df.duplicated(subset=cols).any()


nulos_productos = products.calidad.nulos()
dup_reviews = reviews.calidad.duplicados(subset=["order_id"])
llave_items = items.calidad.llave_unica(["order_id", "order_item_id"])
nulos_productos, dup_reviews, llave_items

(                            n       pct
 product_category_name       6  2.027027
 product_name_lenght         6  2.027027
 product_description_lenght  6  2.027027
 product_photos_qty          6  2.027027,
 6,
 True)

*Nota:* `keep=False` marca **todas** las copias (3 pedidos × 2 reseñas = 6 filas). Con el default `keep="first"` contarías sólo las repeticiones (3). Ambas son útiles; lo importante es que el nombre del método deje claro cuál devuelve.

### Solución 2

In [24]:
tabla_estados = (
  items
  .merge(orders[["order_id", "customer_id"]], on="order_id", validate="m:1")
  .merge(customers[["customer_id", "customer_state"]], on="customer_id", validate="m:1")
  .assign(regiao=lambda d: d["customer_state"].astype(str).br.regiao())
  .groupby("regiao", observed=True)
  .agg(pedidos=("order_id", "nunique"), ingreso=("price", "sum"))
  .assign(ticket=lambda d: d["ingreso"] / d["pedidos"],
          participacion=lambda d: d["ingreso"] / d["ingreso"].sum())
  .sort_values("ingreso", ascending=False)
)

reais = lambda v: pd.Series([v]).br.reais().iloc[0]

sty_estados = (
  tabla_estados.style
  .format({"ingreso": reais, "ticket": reais, "participacion": "{:.1%}"})
  .bar(subset=["participacion"], color="#a9cce3")
  .set_caption("Ingreso por región")
)
sty_estados

,pedidos,ingreso,ticket,participacion
regiao,,,,
Sudeste,204,"R$ 25.498,02","R$ 124,99",69.7%
Sul,49,"R$ 4.522,62","R$ 92,30",12.4%
Nordeste,27,"R$ 4.059,33","R$ 150,35",11.1%
Centro-Oeste,14,"R$ 2.151,61","R$ 153,69",5.9%
Norte,5,"R$ 367,78","R$ 73,56",1.0%


### Solución 3

In [25]:
def hechos_items(df_orders):
  agg = items.groupby("order_id").agg(
    n_items=("order_item_id", "size"),
    valor_items=("price", "sum"),
    flete=("freight_value", "sum"),
  )
  return df_orders.merge(agg, on="order_id", how="left", validate="1:1").fillna(
    {"n_items": 0, "valor_items": 0, "flete": 0}).astype({"n_items": "int64"})


def hechos_pagos(df):
  principal = (payments.sort_values("payment_value", ascending=False)
                       .drop_duplicates("order_id")[["order_id", "payment_type"]]
                       .rename(columns={"payment_type": "metodo_principal"}))
  agg = (payments.groupby("order_id")
                 .agg(pagado=("payment_value", "sum"), cuotas_max=("payment_installments", "max"))
                 .reset_index()
                 .merge(principal, on="order_id", validate="1:1"))
  return df.merge(agg, on="order_id", how="left", validate="1:1")


def hechos_resena(df):
  ultima = (reviews.sort_values("review_answer_timestamp")
                   .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]])
  return df.merge(ultima, on="order_id", how="left", validate="1:1")


def hechos_cliente(df):
  return (df.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left", validate="m:1")
            .assign(regiao=lambda d: d["customer_state"].astype(str).br.regiao()))


fact = (
  orders
  .pedidos.con_tiempos()                    # dias_entrega, tarde, mes  (accessor de la sección 1)
  .pipe(hechos_items)
  .pipe(hechos_pagos)
  .pipe(hechos_resena)
  .pipe(hechos_cliente)
  [["order_id", "order_status", "mes", "customer_state", "regiao", "n_items", "valor_items",
    "flete", "pagado", "metodo_principal", "cuotas_max", "review_score", "dias_entrega", "tarde"]]
)
fact.head()

,order_id,order_status,mes,customer_state,regiao,n_items,valor_items,flete,pagado,metodo_principal,cuotas_max,review_score,dias_entrega,tarde
0,b9a6c5f5df52c7226ac85aee7524c27f,delivered,2018-06,RS,Sul,1,99.00,18.79,117.79,credit_card,2,5.0,6.691944,0.0
1,261e71d2349c713eafa9f3df5972b95d,delivered,2018-01,MG,Sudeste,1,157.77,14.85,172.62,credit_card,10,5.0,9.969884,0.0
2,67b50899f52995848c427e361e10dde3,delivered,2018-06,PR,Sul,1,90.00,18.43,108.43,credit_card,3,1.0,10.662002,0.0
3,32733fc014b67ef70fa6039dd8c6ba82,delivered,2017-08,RJ,Sudeste,1,229.99,26.06,256.05,boleto,1,5.0,25.861748,1.0
4,39a70e9e9b729b11dee34ac12478597f,delivered,2017-08,PR,Sul,1,108.90,20.00,128.90,credit_card,6,3.0,11.804572,0.0


*Notas:*

- Cada etapa agrega su tabla a nivel pedido y une con `validate="1:1"`: imposible inflar totales.
- `metodo_principal` usa el patrón "ordenar + `drop_duplicates`" (Módulos 03 y 05) para quedarse con el pago de mayor valor.
- Reutilizamos el accessor `.pedidos.con_tiempos()` como primera etapa: así los accessors y `pipe` conviven en la misma cadena.

### Solución 4

In [26]:
meses = pd.period_range(fact["mes"].min(), fact["mes"].max(), freq="M")

kpi_mes = (
  fact.groupby("mes")
      .agg(pedidos=("order_id", "size"), ingreso=("valor_items", "sum"),
           pct_tarde=("tarde", "mean"), score=("review_score", "mean"))
      .reindex(meses)
      .fillna({"pedidos": 0, "ingreso": 0})
      .astype({"pedidos": "int64"})
      .assign(ticket=lambda d: d["ingreso"] / d["pedidos"].replace(0, np.nan),
              pct_tarde=lambda d: d["pct_tarde"] * 100,
              crec_ingreso=lambda d: d["ingreso"].pct_change() * 100)
      [["pedidos", "ingreso", "ticket", "pct_tarde", "score", "crec_ingreso"]]
      .rename_axis("mes")
)

kpi_region_trim = (
  fact.assign(trimestre=lambda d: d["mes"].dt.asfreq("Q").astype(str))
      .pivot_table(index="regiao", columns="trimestre", values="valor_items",
                   aggfunc="sum", fill_value=0, observed=True)
      .assign(Total=lambda d: d.sum(axis=1))
      .sort_values("Total", ascending=False)
      .rename_axis(index="regiao", columns=None)
)

peor_mes_score = str(kpi_mes.query("pedidos >= 10")["score"].idxmin())
kpi_mes.round(1).tail(6), kpi_region_trim.round(0), peor_mes_score

(         pedidos  ingreso  ticket  pct_tarde  score  crec_ingreso
 mes                                                              
 2018-03       23   3146.9   136.8       30.4    3.7          21.3
 2018-04       21   4121.0   196.2        0.0    4.2          31.0
 2018-05       21   3620.3   172.4        4.8    4.7         -12.1
 2018-06       25   3045.9   121.8        4.0    4.0         -15.9
 2018-07       15   1389.4    92.6        6.7    4.4         -54.4
 2018-08       21   2621.7   124.8        5.0    4.2          88.7,
               2017Q1  2017Q2  2017Q3  2017Q4  2018Q1  2018Q2  2018Q3    Total
 regiao                                                                       
 Sudeste       1732.0  1802.0  1931.0  4077.0  5951.0  7547.0  2458.0  25498.0
 Sul             99.0   269.0   431.0   268.0  1256.0  1366.0   834.0   4523.0
 Nordeste        80.0   160.0   322.0   918.0  1225.0   923.0   431.0   4059.0
 Centro-Oeste     0.0    85.0   225.0   372.0   350.0   833.0   288.

*Notas:* `reindex(meses)` garantiza los 20 meses aunque falten algunos (Módulo 06). `ticket` divide entre `pedidos` con 0 → `NaN` para no obtener infinitos. `dt.asfreq("Q")` convierte un `Period` mensual a su trimestre.

### Solución 5

In [27]:
ruta_pq = SALIDA / "fact_pedidos.parquet"
fact.to_parquet(ruta_pq, index=False)
fact_releido = pd.read_parquet(ruta_pq)
print(fact_releido.dtypes[["mes", "customer_state", "regiao"]])

trimestres = [c for c in kpi_region_trim.columns if c != "Total"]
ruta_rep = SALIDA / "reporte_olist.xlsx"

with pd.ExcelWriter(ruta_rep, engine="openpyxl") as xl:
  (kpi_mes.set_axis(kpi_mes.index.astype(str))
          .style
          .format({"ingreso": "{:,.2f}", "ticket": "{:,.2f}", "pct_tarde": "{:.1f}",
                   "score": "{:.2f}", "crec_ingreso": "{:.1f}"}, na_rep="")
          .background_gradient(subset=["pct_tarde"], cmap="Reds")
          .to_excel(xl, sheet_name="kpis_mensuales"))
  (kpi_region_trim.style
          .format("{:,.0f}")
          .background_gradient(subset=trimestres, cmap="Blues", axis=None)
          .to_excel(xl, sheet_name="region_trimestre"))
  (por_cat.style
          .format({"ingreso": "{:,.2f}", "ticket": "{:,.2f}", "pct_flete": "{:.1%}",
                   "pct_tarde": "{:.1%}", "participacion": "{:.1%}"})
          .bar(subset=["ingreso"], color="#7fb3d5")
          .to_excel(xl, sheet_name="top_categorias"))

hojas = pd.ExcelFile(ruta_rep, engine="openpyxl").sheet_names
hojas

mes               period[M]
customer_state     category
regiao             category
dtype: object


['kpis_mensuales', 'region_trimestre', 'top_categorias']

*Notas:*

- Parquet conserva `Period` y `category` (gracias a los metadatos de pandas que guarda pyarrow); un CSV los perdería.
- `set_axis(index.astype(str))` convierte el `PeriodIndex` a texto: Excel no tiene un tipo "periodo".
- `background_gradient(..., axis=None)` usa una sola escala para toda la tabla de trimestres (en lugar de una por columna), que es lo que quieres para comparar celdas entre sí.

---
# 🎓 Cierre del curso

Recorriste:

| Módulo | Idea clave |
|---|---|
| 01 | Declara los dtypes al leer; pandas 3 = `str` + Copy-on-Write |
| 02 | El índice es una llave; las operaciones se alinean por etiquetas |
| 03 | `agg` / `transform` / `filter` / `apply`, y cuándo usar cada uno |
| 04 | Calcula en largo, presenta en ancho |
| 05 | Revisa cardinalidad, valida y agrega antes de unir |
| 06 | Regulariza el tiempo antes de usar ventanas y rezagos |
| 07 | Llaves canónicas y validación como reglas |
| 08 | Cadenas legibles y pipelines por etapas |
| 09 | Mide, vectoriza, elige dtypes y formatos eficientes |
| 10 | Empaqueta tu dominio (accessors) y presenta con `Styler` |

### Siguientes pasos sugeridos

- Repetir el proyecto final con el **dataset completo** de Olist (~100 mil pedidos): los mismos notebooks funcionan cambiando la carpeta `data/`, y verás los efectos de rendimiento con datos reales.
- Mover tus funciones de etapa y accessors a un módulo `.py` con **pruebas** (`pytest`).
- Explorar herramientas que complementan a pandas para datos más grandes que la memoria (DuckDB, Polars), sabiendo ya qué problemas resuelven.